# NestedKriging on the Branin 2D function (Julia)

This notebook demonstrates **NestedKriging**, a divide-and-conquer Gaussian Process for
large designs: the data are partitioned into $p$ groups, one Kriging submodel is fitted
per group (all sharing a **common prior** after hyperparameter unification), and
predictions are aggregated.

Available aggregations:
- **NK** (default): the optimal aggregation of Rullière, Durrande, Bachoc & Chevalier
  (*Statistics & Computing*, 2018) — itself a kriging predictor: it interpolates the
  design and provides consistent variances;
- **PoE / gPoE / BCM / rBCM**: precision-weighted products of experts (cheaper, but
  no interpolation guarantee).

Fit cost drops from $O(n^3)$ to $O(n^3/p^2)$ per likelihood evaluation.

Steps:
1. Setup
2. Branin function
3. Design of experiments ($n = 800$)
4. Fit `NestedKriging` and inspect the partition
5. Predict on a grid: mean, stdev, RMSE, interpolation check
6. Compare aggregations
7. Warped variant (WarpKriging submodels)


## 0. Setup

Build the C++ core and install the Julia binding (skip if already done).
Requires: `cmake`, a C++ compiler, Julia ≥ 1.10.

```shell
julia -e 'using Pkg; Pkg.develop(path="bindings/Julia/jlibkriging")'
```

In [ ]:
using jlibkriging
using Plots
using Random

println("jlibkriging loaded")

## 2. Branin function

In [ ]:
function branin(X::Matrix{Float64})
    x1 = X[:, 1] .* 15 .- 5
    x2 = X[:, 2] .* 15
    return (x2 .- 5 ./ (4π^2) .* x1.^2 .+ 5 ./ π .* x1 .- 6).^2 .+
           10 .* (1 - 1 / (8π)) .* cos.(x1) .+ 10
end

grid_x = collect(range(0, 1, length=50))
grid_pts = hcat(vec([x1 for x2 in grid_x, x1 in grid_x]),
                vec([x2 for x2 in grid_x, x1 in grid_x]))
z_true = branin(grid_pts)

contourf(grid_x, grid_x, reshape(z_true, 50, 50), title="True Branin")

## 3. Design of experiments (n = 800)

In [ ]:
Random.seed!(123)
n = 800
X = rand(n, 2)
y = branin(X)
size(X)

## 4. Fit NestedKriging (NK aggregation, 8 groups)

In [ ]:
nk = NestedKriging(y, X, "matern5_2", 8)  # aggregation="NK", partition="kmeans" by default
println(summary(nk))

## 5. Prediction on the grid

In [ ]:
p = predict(nk, grid_pts)

display(contourf(grid_x, grid_x, reshape(p.mean, 50, 50), title="NK mean"))
display(contourf(grid_x, grid_x, reshape(p.stdev, 50, 50), title="NK stdev"))

println("RMSE vs true Branin : ", sqrt(sum((p.mean .- z_true).^2) / length(z_true)))
pX = predict(nk, X)
println("max |mean - y| at design points (interpolation): ", maximum(abs.(pX.mean .- y)))

## 6. Comparing aggregations

In [ ]:
for agg in ["NK", "gPoE", "rBCM", "PoE", "BCM"]
    k = NestedKriging(y, X, "matern5_2", 8; aggregation=agg)
    m = predict(k, grid_pts; return_stdev=false)
    rmse = sqrt(sum((m.mean .- z_true).^2) / length(z_true))
    println(rpad(agg, 5), " RMSE = ", round(rmse, digits=4))
end

## 7. Warped variant

In [ ]:
nkw = NestedKriging(y, X, "gauss", 8; warping=["kumaraswamy", "kumaraswamy"])
pw = predict(nkw, grid_pts)
println("RMSE (warped): ", sqrt(sum((pw.mean .- z_true).^2) / length(z_true)))

## Notes

- The NK aggregation **interpolates** whatever the hyperparameters: it is a property of
  the aggregation itself, not of the fit.
- NK requires a **constant trend**; use the PoE family for other trends.
- Memory/speed of the NK prediction can be tuned with `set_predict_chunk`.
- In the warped case, the prior hyperparameters $(\theta, \text{warp})$ are estimated by a
  **single reference fit on a global subsample** (default 1000 points, see
  `set_warp_subsample`), then submodels are fitted in closed form (`optim="none"`).
